# KCS early-20 HWPX current source metadata probe
Two exact official attachment reads; no market outcomes.


In [1]:
import datetime,hashlib,json,re,shlex,subprocess,sys,tarfile,tempfile,zipfile
from pathlib import Path
from xml.etree import ElementTree as ET
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,_settings,_ssh
card=root/'research/kcs_early20_hwpx_vintage_probe_20261023';raw=root/'data/snapshots/kcs_early20_hwpx_vintage_probe_20261023';raw.mkdir(parents=True,exist_ok=True);plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
for key,hkey in [('2025_PDF','2025_PDF_review_sha256'),('2026_PDF','2026_PDF_review_sha256')]:assert sha(root/plan['reference_reviews'][key])==plan['reference_reviews'][hkey]
(card/'run_state.json').write_text(json.dumps({'status':'started','plan_sha256':sha(card/'plan.json'),'GETs_started':0},indent=2)+'\n')
receipts=[];inspection={}
for count,(name,url) in enumerate(plan['source_urls'].items(),1):
 body=raw/(name+'.body');headers=raw/(name+'.headers');start=datetime.datetime.now(datetime.timezone.utc).isoformat()
 call=subprocess.run(['curl','--location','--silent','--show-error','--connect-timeout','5','--max-time','20','--dump-header',str(headers),'--output',str(body),'--write-out','%{http_code}|%{url_effective}',url],capture_output=True,text=True)
 end=datetime.datetime.now(datetime.timezone.utc).isoformat();parts=call.stdout.strip().split('|',1)
 receipt={'source_name':name,'source_url':url,'started_at_utc':start,'completed_at_utc':end,'curl_exit':call.returncode,'http_status':parts[0] if parts else None,'effective_url':parts[1] if len(parts)>1 else None,'error':call.stderr[:300] if call.returncode else None,'body_path':str(body.relative_to(root)),'body_bytes':body.stat().st_size if body.exists() else 0,'body_sha256':sha(body) if body.exists() else None,'headers_path':str(headers.relative_to(root)),'headers_bytes':headers.stat().st_size if headers.exists() else 0,'headers_sha256':sha(headers) if headers.exists() else None}
 receipts.append(receipt);(card/'run_state.json').write_text(json.dumps({'status':'capturing','plan_sha256':sha(card/'plan.json'),'GETs_started':count},indent=2)+'\n')
 valid=receipt['http_status']=='200' and body.exists() and zipfile.is_zipfile(body)
 detail={'valid_HWPX_ZIP':valid,'entry_count':None,'entry_names':[],'zip_date_range':None,'metadata_snippets':[],'numeric_token_presence':{},'parse_error':None}
 if valid:
  try:
   with zipfile.ZipFile(body) as archive:
    infos=archive.infolist();total=sum(x.file_size for x in infos)
    if len(infos)>1000 or total>20000000:raise ValueError('HWPX decompression safety limit')
    detail['entry_count']=len(infos);detail['entry_names']=[x.filename for x in infos[:80]];dates=[x.date_time for x in infos if x.date_time[0]>=1980];detail['zip_date_range']=[min(dates),max(dates)] if dates else None
    chunks=[]
    for entry in infos:
     if not entry.filename.lower().endswith('.xml'):continue
     data=archive.read(entry)
     if len(data)>3000000:continue
     try:element=ET.fromstring(data)
     except ET.ParseError:continue
     txt=' '.join(t.strip() for t in element.itertext() if t and t.strip());chunks.append(txt)
     if any(word in entry.filename.lower() for word in ['meta','core','property']):
      detail['metadata_snippets'].append({'entry':entry.filename,'text':txt[:1000]})
    alltext=' '.join(chunks)
    for token in (['8,533','8533','20.2'] if name.startswith('2025') else ['27,516','27516','222.5']):detail['numeric_token_presence'][token]=token in alltext
  except Exception as err:detail['parse_error']=type(err).__name__+': '+str(err)[:250]
 inspection[name]=detail
(card/'response_receipts.json').write_text(json.dumps(receipts,ensure_ascii=False,indent=2)+'\n')
result={'status':'two_official_current_HWPX_bytes_inspected_first_public_unqualified' if all(x['valid_HWPX_ZIP'] and not x['parse_error'] for x in inspection.values()) else 'needs_HWPX_transport_or_parse','inspection':inspection,'GETs':len(receipts),'historical_first_public_bytes_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'strategy_forward_days':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
entries={str(p.relative_to(root)):sha(p) for p in sorted(raw.iterdir()) if p.is_file()};entries[str((card/'plan.json').relative_to(root))]=sha(card/'plan.json');entries[str((card/'response_receipts.json').relative_to(root))]=sha(card/'response_receipts.json')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='kcs-hwpx-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for rel in sorted(entries):out.add(root/rel,arcname=rel,recursive=False)
 digest=sha(archive);size=archive.stat().st_size;evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert evidence['sha256']==digest and evidence['size_bytes']==size
 nas_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(nas_path)).split()[0];assert remote==digest
 with tempfile.TemporaryDirectory(prefix='kcs-hwpx-readback-') as back:
  fetched=Path(back)/'restore.tar.gz'
  with fetched.open('wb') as output:
   proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'cat '+shlex.quote(nas_path)],stdout=output,stderr=subprocess.PIPE,timeout=90)
  assert proc.returncode==0 and sha(fetched)==digest
  with tarfile.open(fetched,'r:gz') as source:
   members={m.name:m for m in source.getmembers()};assert set(entries)==set(members)
   for rel,expected in entries.items():
    f=source.extractfile(members[rel]);assert f and hashlib.sha256(f.read()).hexdigest()==expected
receipt={'status':'uploaded_exact_current_HWPX_source_bundle','archive_sha256':digest,'archive_bytes':size,'entries':entries,'nas_artifact':evidence,'nas_path':nas_path,'remote_hash_verified':True,'scoped_restore_verified':True}
(card/'source_archive_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','source_NAS_sha256':digest,'source_NAS_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');(card/'run_state.json').write_text(json.dumps({'status':'completed','plan_sha256':sha(card/'plan.json'),'GETs_completed':len(receipts),'NAS_scoped_restore_verified':True},indent=2)+'\n')
print(json.dumps({'result':result,'archive_sha256':digest},ensure_ascii=False))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{"result": {"status": "two_official_current_HWPX_bytes_inspected_first_public_unqualified", "inspection": {"2025_Oct_early20_HWPX": {"valid_HWPX_ZIP": true, "entry_count": 34, "entry_names": ["mimetype", "version.xml", "Contents/header.xml", "BinData/image1.jpg", "BinData/image2.jpg", "BinData/image3.wmf", "BinData/image4.wmf", "BinData/image5.wmf", "BinData/image6.wmf", "BinData/image7.wmf", "BinData/image8.wmf", "BinData/image9.wmf", "BinData/image10.wmf", "BinData/image11.wmf", "BinData/image12.wmf", "BinData/image13.wmf", "BinData/image14.wmf", "BinData/image15.wmf", "BinData/image16.wmf", "BinData/image17.wmf", "BinData/image18.wmf", "BinData/image19.wmf", "BinData/image20.wmf", "BinData/image21.jpg", "BinData/image22.jpg", "BinData/image23.jpg", "Contents/section0.xml", "Preview/PrvText.txt", "settings.xml", "Preview/PrvImage.png", "META-INF/container.rdf", "Contents/content.hpf", "META-INF/container.xml", "META-INF/manifest.xml"], "zip_date_range": [[1980, 1, 1, 0, 0, 0], [1980,

# 韩国海关前20日 HWPX 当前原件时序预检

按冻结计划分别抓取官方门户所列的 2025、2026 年 10 月 1–20 日 HWPX 附件各一次，均为 HTTP 200 和有效 ZIP/HWPX：2025 年 811,885 字节，2026 年 795,926 字节。原件正文 XML 分别含半导体 **8,533 百万美元、+20.2%** 与 **27,516 百万美元、+222.5%**，与已视觉核对的相应 PDF 数值一致。响应、SHA-256 及原始字节 NAS 归档 `2cae4e98199e7704fd911f63fae67c7b137d0f0b5d745ae7435519ca1d4a5017` 已通过远端哈希和限定恢复。

内部 `Contents/content.hpf` 的 `ModifiedDate` 分别为 **2025-10-21 07:40:04** 和 **2026-10-21 08:30:25 韩国时间**，都早于各自印刷的 **09:00** 分发时刻；与 2025 年当前 PDF 次日才创建的情形不同，HWPX 是更值得追查的原版候选。但两份 HWPX 的 ZIP 成员时间全为 1980 年占位值、`CreatedDate` 都沿用 2008 年模板、另一个日期栏也不对应发布；内部可改的 `ModifiedDate` **不证明这些精确字节当时已公开或未被后改**。详见 `internal_metadata_review.json`。

本卡只提升来源语义与原版候选的可追查性，**没有取得历史首发收据**。0 SOXX 结果读取、0 拟合、0 账户、0 策略前向日。下一步若要历史特征，须找到独立的当时下载收据或不可变归档；11 月正式月报的前向原始字节采集仍按已冻结计划运行。
